In [1]:
import pandas as pd
import requests
import zipfile
from pathlib import Path

PASTA_RAW   = Path("../data/raw")
PASTA_CLEAN = Path("../data/clean")

def baixar_arquivo(url, destino):
    destino = Path(destino)
    if destino.exists():
        print(f"Já existe, pulando: {destino.name}")
        return destino
    print(f"Baixando {destino.name}...")
    resposta = requests.get(url, stream=True, timeout=180)
    resposta.raise_for_status()
    with open(destino, "wb") as f:
        for pedaco in resposta.iter_content(chunk_size=1024 * 1024):
            f.write(pedaco)
    print(f"Concluído: {destino.stat().st_size / 1_000_000:.0f} MB")
    return destino

URL_2014 = "https://download.inep.gov.br/microdados/microdados_censo_da_educacao_superior_2014.zip"
zip_2014 = baixar_arquivo(URL_2014, PASTA_RAW / "censo_2014.zip")

Já existe, pulando: censo_2014.zip


In [3]:
pasta_censo_2014 = PASTA_RAW / "censo_2014"

csvs_2014 = (
    list(pasta_censo_2014.rglob("*.CSV")) +
    list(pasta_censo_2014.rglob("*.csv"))
)

print(f"CSVs encontrados em censo_2014:")
for i, csv in enumerate(csvs_2014):
    tamanho_mb = csv.stat().st_size / 1_000_000
    print(f"  [{i}]  {csv.name}  —  {tamanho_mb:.0f} MB")

CSVs encontrados em censo_2014:
  [0]  MICRODADOS_CADASTRO_IES_2014.CSV  —  1 MB
  [1]  MICRODADOS_CADASTRO_CURSOS_2014.CSV  —  43 MB


In [4]:
caminho_csv_2014 = csvs_2014[1]

cabecalho_2014 = pd.read_csv(
    caminho_csv_2014,
    nrows=0,
    sep=";",
    encoding="latin-1"
)

print(f"Total de colunas em 2014: {len(cabecalho_2014.columns)}")

colunas_que_precisamos = [
    "TP_MODALIDADE_ENSINO",
    "CO_REGIAO",
    "TP_GRAU_ACADEMICO",
    "TP_REDE",
    "QT_MAT",
    "QT_CONC",
    "QT_MAT_18_24",
    "QT_MAT_25_29",
    "QT_MAT_30_34",
]

print("\nVerificando colunas necessárias:")
for col in colunas_que_precisamos:
    status = "OK" if col in cabecalho_2014.columns else "AUSENTE"
    print(f" {status} {col}")

Total de colunas em 2014: 200

Verificando colunas necessárias:
 OK TP_MODALIDADE_ENSINO
 OK CO_REGIAO
 OK TP_GRAU_ACADEMICO
 OK TP_REDE
 OK QT_MAT
 OK QT_CONC
 OK QT_MAT_18_24
 OK QT_MAT_25_29
 OK QT_MAT_30_34


In [6]:
COLUNAS = [
    "TP_MODALIDADE_ENSINO",
    "CO_REGIAO",
    "TP_GRAU_ACADEMICO",
    "TP_REDE",
    "QT_MAT",
    "QT_CONC",
    "QT_MAT_18_24",
    "QT_MAT_25_29",
    "QT_MAT_30_34",
]

df_2014 = pd.read_csv(
    caminho_csv_2014,
    sep=";",
    encoding="latin-1",
    usecols=COLUNAS,
    low_memory=False
)

print(f"Linhas carregadas: {len(df_2014):,}")
df_2014.head()

Linhas carregadas: 73,569


,CO_REGIAO,TP_REDE,TP_GRAU_ACADEMICO,TP_MODALIDADE_ENSINO,QT_MAT,QT_MAT_18_24,QT_MAT_25_29,QT_MAT_30_34,QT_CONC
0,1.0,1,1.0,1,78.0,52.0,15.0,5.0,0.0
1,1.0,1,2.0,1,186.0,64.0,48.0,35.0,22.0
2,1.0,2,1.0,1,47.0,27.0,12.0,2.0,12.0
3,1.0,2,2.0,1,31.0,16.0,8.0,3.0,0.0
4,1.0,2,1.0,1,240.0,127.0,65.0,31.0,75.0


In [14]:
resumo_2024 = pd.read_csv(PASTA_CLEAN / "resumo_modalidade_2024.csv")

resumo_2014 = (
    df_2014.groupby("TP_MODALIDADE_ENSINO")[["QT_MAT", "QT_CONC"]]
    .sum()
    .reset_index()
)
resumo_2014["modalidade"] = resumo_2014["TP_MODALIDADE_ENSINO"].map({
    1: "Presencial", 2: "EAD"
})
total_2014 = resumo_2014["QT_MAT"].sum()
resumo_2014["share_pct"] = (resumo_2014["QT_MAT"] / total_2014 * 100).round(1)
resumo_2014["taxa_conclusao_pct"] = (
    resumo_2014["QT_CONC"] / resumo_2014["QT_MAT"] * 100
).round(1)
resumo_2014["ano"] = 2014

resumo_2024["ano"] = 2024
total_2024 = resumo_2024["QT_MAT"].sum()
resumo_2024["share_pct"] = (
    resumo_2024["QT_MAT"] / total_2024 * 100
).round(1)

comparacao = pd.concat([resumo_2014, resumo_2024], ignore_index=True)

print("=" * 60)
print("COMPARAÇÃO 2014 vs 2024")
print("=" * 60)

for modalidade in ["Presencial", "EAD"]:
    print(f"\n── {modalidade} ──")
    subset = comparacao[comparacao["modalidade"] == modalidade][
        ["ano", "QT_MAT", "share_pct", "taxa_conclusao_pct"]
    ].sort_values("ano")

    for _, row in subset.iterrows():
        print(f"  {int(row['ano'])}  "
              f"Matrículas: {row['QT_MAT']:>10,.0f}  "
              f"Share: {row['share_pct']:>5.1f}%  "
              f"Conclusão: {row['taxa_conclusao_pct']}%")

print("\n── Variação no período ──")
for modalidade in ["Presencial", "EAD"]:
    mat_2014 = comparacao[
        (comparacao["modalidade"] == modalidade) &
        (comparacao["ano"] == 2014)
    ]["QT_MAT"].values[0]

    mat_2024 = comparacao[
        (comparacao["modalidade"] == modalidade) &
        (comparacao["ano"] == 2024)
    ]["QT_MAT"].values[0]

    variacao_pct = (mat_2024 / mat_2014 - 1) * 100
    variacao_abs = mat_2024 - mat_2014

    print(f"  {modalidade:12}  "
          f"Variação absoluta: {variacao_abs:>+10,.0f}  "
          f"Variação percentual: {variacao_pct:>+.1f}%")

total_var = (total_2024 / total_2014 - 1) * 100
print(f"\n  {'TOTAL':12}  "
      f"2014: {total_2014:>10,.0f}  "
      f"2024: {total_2024:>10,.0f}  "
      f"Variação: {total_var:>+.1f}%")
    
    


COMPARAÇÃO 2014 vs 2024

── Presencial ──
  2014  Matrículas:  6,497,889  Share:  82.9%  Conclusão: 12.9%
  2024  Matrículas:  5,037,875  Share:  49.3%  Conclusão: 14.5%

── EAD ──
  2014  Matrículas:  1,341,876  Share:  17.1%  Conclusão: 14.1%
  2024  Matrículas:  5,189,391  Share:  50.7%  Conclusão: 11.7%

── Variação no período ──
  Presencial    Variação absoluta: -1,460,014  Variação percentual: -22.5%
  EAD           Variação absoluta: +3,847,515  Variação percentual: +286.7%

  TOTAL         2014:  7,839,765  2024: 10,227,266  Variação: +30.5%


In [16]:
comparacao.to_csv(PASTA_CLEAN / "comparacao_2014_2024.csv", index=False)

numeros_post = {
    "total_2014": int(total_2014),
    "total_2024": int(total_2024),
    "variacao_total_pct": round(total_var, 1),
    "presencial_2014": 6_497_889,
    "presencial_2024": 5_037_875,
    "variacao_presencial_pct": -22.5,
    "ead_2014": 1_341_876,
    "ead_2024": 5_189_391,
    "variacao_ead_pct": 286.7,
    "estudantes_novos_estimativa": 3_847_515 - 1_460_014,
    "share_ead_2014_pct": 17.1,
    "share_ead_2024_pct": 50.7,
    "conclusao_presencial_2014_pct": 12.9,
    "conclusao_presencial_2024_pct": 14.5,
    "conclusao_ead_2014_pct": 14.1,
    "conclusao_ead_2024_pct": 11.7,
}

import json
with open(PASTA_CLEAN / "numeros_post_ead.json", "w") as f:
    json.dump(numeros_post, f, indent=2, ensure_ascii = False)

print("Arquivos salvos:")
print(" comparacao_2014_2024.csv")
print(" numeros_post_ead.json")
print("\nTodos os números têm fonte: INEP, Censo da Educação Superior 2014 e 2024")
    

Arquivos salvos:
 comparacao_2014_2024.csv
 numeros_post_ead.json

Todos os números têm fonte: INEP, Censo da Educação Superior 2014 e 2024
